In [1]:
import os, glob, unicodedata
import pandas as pd
import numpy as np

d = '../datos/raw/'

def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return ""
    s = ''.join(c for c in unicodedata.normalize('NFD', str(nombre)) if unicodedata.category(c) != 'Mn')
    return s.strip().lower()

def cargar_fbref(tipo):
    lista = []
    for f in sorted(glob.glob(f'{d}fbref_{tipo}_*.csv')):
        s = os.path.basename(f).replace(f'fbref_{tipo}_', '').replace('.csv', '')
        x = pd.read_csv(f, header=1, on_bad_lines='skip')
        x = x[x['Player'].notna() & (x['Player'] != 'Player')].copy()
        x['Season'] = s
        lista.append(x)
    return pd.concat(lista, ignore_index=True)

In [2]:
fb = cargar_fbref('std')
for c in ['Min', 'Gls', 'Ast']:
    fb[c] = pd.to_numeric(fb[c].astype(str).str.replace(',', ''), errors='coerce')
fb = fb.dropna(subset=['Min'])
fb['Player_Clean'] = fb['Player'].apply(limpiar_nombre)

# En tus archivos Gls y Ast vienen por 90: reconstruimos los totales
fb['G_tot'] = (fb['Gls'] * fb['Min'] / 90).round()
fb['A_tot'] = (fb['Ast'] * fb['Min'] / 90).round()
print(len(fb), 'filas FBref;', fb['Season'].nunique(), 'temporadas')

3352 filas FBref; 6 temporadas


In [3]:
lista = []
for f in sorted(glob.glob(d + 'under_2*.csv')):
    s = os.path.basename(f).replace('under_', '').replace('.csv', '')
    x = pd.read_csv(f, sep=None, engine='python')
    x['Season'] = s
    lista.append(x)
us = pd.concat(lista, ignore_index=True)
us['Player_Clean'] = us['player'].apply(limpiar_nombre)

print(us.columns.tolist())
print('Duplicados jugador-temporada en Understat:', us.duplicated(['Player_Clean', 'Season']).sum())

['\ufeff"number"', 'player', 'team', 'apps', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xGChain', 'xGBuildup', 'xG90', 'NPxG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90', 'Season', 'Player_Clean']
Duplicados jugador-temporada en Understat: 3


In [4]:
es_fw = fb['Pos'].str.startswith('FW', na=False)
fw_keys = set(zip(fb.loc[es_fw, 'Player_Clean'], fb.loc[es_fw, 'Born']))

agg = (fb.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
         .agg(Player=('Player', 'first'),
              Squad=('Squad', lambda x: '/'.join(x)),
              Min=('Min', 'sum'),
              G_tot=('G_tot', 'sum'),
              A_tot=('A_tot', 'sum')))

agg['es_fw'] = [(p, b) in fw_keys for p, b in zip(agg['Player_Clean'], agg['Born'])]
agg = agg[agg['es_fw'] & (agg['Min'] >= 900)].copy()
print('Registros jugador-temporada (delanteros, >=900 min):', len(agg))

Registros jugador-temporada (delanteros, >=900 min): 432


In [10]:
equivalencias = {
    'emi buendia': 'emiliano buendia',
    'francisco trincao': 'trincao',
    'hwang hee-chan': 'hee-chan hwang',
    'igor thiago': 'thiago',
    'josh sargent': 'joshua sargent',
}

agg['Player_Clean_fbref'] = agg['Player_Clean']
agg['Player_Clean'] = agg['Player_Clean'].replace(equivalencias)

cruce = agg.merge(us, on=['Player_Clean', 'Season'], how='left', indicator=True)
print(cruce['_merge'].value_counts())
print('Filas en agg:', len(agg), '| filas tras el cruce:', len(cruce))

sin_cruce = cruce[cruce['_merge'] == 'left_only']
print(sin_cruce[['Player', 'Squad', 'Season', 'Min']].to_string())

_merge
both          432
left_only       0
right_only      0
Name: count, dtype: int64
Filas en agg: 432 | filas tras el cruce: 432
Empty DataFrame
Columns: [Player, Squad, Season, Min]
Index: []


In [ ]:
ok = cruce.copy()
ok['ratio_min'] = ok['min'] / ok['Min']
ok['dif_goles'] = (ok['goals'] - ok['G_tot']).abs()

sospechosos = ok[(ok['ratio_min'] < 0.85) | (ok['ratio_min'] > 1.15) | (ok['dif_goles'] > 3)]
print(len(sospechosos), 'registros a revisar de', len(ok))
print(sospechosos[['Player', 'Squad', 'team', 'Season', 'Min', 'min', 'G_tot', 'goals']].to_string())

In [ ]:
cols = ['Player', 'Player_Clean', 'Born', 'Season', 'Squad', 'Min', 'G_tot', 'A_tot',
        'team', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xG90', 'NPxG90', 'xA90']
cols = [c for c in cols if c in cruce.columns]

base = cruce[cols].copy()
os.makedirs('../datos/procesado', exist_ok=True)
base.to_csv('../datos/procesado/fw_base.csv', index=False)
print(base.shape)
display(base.head())
sht = cargar_fbref('sht')
print(sht.columns.tolist())
print(sht[sht['Player'] == 'Mohamed Salah'].head(3).to_string())

In [16]:
sht = cargar_fbref('sht')
for c in ['90s', 'Sh', 'SoT']:
    sht[c] = pd.to_numeric(sht[c], errors='coerce')
sht['Player_Clean'] = sht['Player'].apply(limpiar_nombre)

# Totales reconstruidos desde las tasas por 90
sht['Sh_tot'] = (sht['Sh'] * sht['90s']).round()
sht['SoT_tot'] = (sht['SoT'] * sht['90s']).round()

sht['Player_Clean'] = sht['Player_Clean'].replace(equivalencias)

sht_agg = (sht.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
              .agg(Sh_tot=('Sh_tot', 'sum'), SoT_tot=('SoT_tot', 'sum')))

In [ ]:
base = cruce.merge(sht_agg, on=['Player_Clean', 'Born', 'Season'],
                   how='left', indicator='_m_sht')
print(base['_m_sht'].value_counts())
print('Filas:', len(cruce), '->', len(base))
print(base[['Sh_tot', 'SoT_tot']].isna().sum())
print('SoT > Sh (debe ser 0):', (base['SoT_tot'] > base['Sh_tot']).sum())

# Todo por 90 con una sola base de minutos (FBref)
n90 = base['Min'] / 90
base['Sh_p90'] = base['Sh_tot'] / n90
base['SoT_p90'] = base['SoT_tot'] / n90
base['npG_p90'] = base['NPG'] / n90
base['NPxG_p90'] = base['NPxG'] / n90
base['xA_p90'] = base['xA'] / n90
base['npG_menos_NPxG_p90'] = (base['NPG'] - base['NPxG']) / n90

crit = ['Sh_p90', 'SoT_p90', 'npG_p90', 'NPxG_p90', 'xA_p90', 'npG_menos_NPxG_p90']
print(base[crit].describe().round(2))

# Control con Salah
print(base.loc[base['Player_Clean'] == 'mohamed salah',
               ['Season', 'Min', 'Sh_tot', 'SoT_tot', 'npG_p90', 'NPxG_p90']].round(2).to_string())